# Chains (LangChain v1.2)

**LCEL(LangChain Expression Language)**을 사용해서 모든 구성 요소가 `Runnable` 인터페이스로 통합되어 파이프라인(`|`)으로 연결될 수 있다.

```python
chain = prompt | model | output_parser  # 기본 구조
```

**구성 요소 업데이트 (v1.2 기준)**
1. **PromptTemplate**  
   - `Runnable`로 변환되어 LCEL 파이프라인에 직접 통합  
   ```python
   prompt = ChatPromptTemplate.from_template("...")
   ```

2. **LLM/ChatModel**  
   - `ChatOpenAI`, `ChatAnthropic` 등이 `Runnable` 구현  
   ```python
   model = ChatOpenAI(model="gpt-4o")
   ```

3. **Memory**  
   - `RunnableWithMessageHistory`로 통합 관리 (또는 LangGraph Persistence 사용)
   ```python
   chain_with_memory = RunnableWithMessageHistory(
       base_chain,
       get_session_history
   )
   ```

4. **Output Parsers**  
   - `StrOutputParser()`, `JsonOutputParser()` 등이 `Runnable`로 작동  
   ```python
   output_parser = JsonOutputParser()
   ```

5. **Tools**  
   - `@tool` 데코레이터로 생성 후 `RunnableLambda`로 변환  
   ```python
   @tool
   def search(query: str) -> str: ...
   ```

**체인 유형별 구현**


1. Simple Chain  

    ```python
    chain = prompt | model | output_parser
    response = chain.invoke({"input": "..."})
    ```

2. Sequential Chain  

    ```python
    chain = (
        {"step1_output": prompt1 | model1}  # 첫 번째 체인 결과 매핑
        | prompt2
        | model2
    )
    ```

3. Conditional Chain
    - `RunnableBranch` 사용

    ```python
    branch = RunnableBranch(
        (lambda x: x["topic"] == "math", math_chain),
        (lambda x: x["topic"] == "history", history_chain),
        default_chain
    )
    ```

4. Memory Chain  

    ```python
    memory_chain = RunnableWithMessageHistory(
        core_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )
    ```


**🚨 v1.2 주요 변경점**

- **Legacy Chain 클래스 완전 폐기**: `LLMChain`, `SequentialChain` 등은 `langchain-classic`으로 이동되거나 삭제됨 → `Runnable` (LCEL)로 통합
- **에이전트 통합**: `create_agent` (LangGraph 기반)가 표준

In [1]:
%pip install -Uqqq langchain langchain-openai langchain-community

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://apac.api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

### Simple Chain

In [3]:
from langchain_core.prompts import PromptTemplate     # prompt chain 구성
from langchain.chat_models import init_chat_model     # 모델 chain 구성 래퍼
from langchain_core.output_parsers import StrOutputParser  # 답변 문자형 변환

prompt = PromptTemplate.from_template('{city}의 특산물은 무엇입니까?')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser
print(chain.invoke('강원도'))  # 프롬프트 템플릿 변수가 1개일 경우만 사용
# 프롬프트 템플릿 변수가 2개 이상일 경우 dict형으로 전달
# print(chain.invoke(input={'city': '강원도', ...}))
print()
print(chain.invoke(input={'city': '강원도'}))

강원도의 대표적인 특산물은 다음과 같습니다.

- **감자**: 강원도 감자, 특히 평창·강릉 지역이 유명합니다.
- **옥수수**: 찰옥수수로 유명하며 홍천·정선 등에서 많이 생산됩니다.
- **메밀**: 봉평 메밀과 메밀국수, 메밀전병이 대표적입니다.
- **황태**: 인제 용대리 황태가 유명합니다.
- **오징어·명태**: 동해안 지역의 대표 수산물입니다.
- **한우**: 횡성한우가 대표적입니다.
- **더덕·곤드레·산나물**: 정선, 홍천 등 산간 지역에서 많이 생산됩니다.
- **잣**: 홍천 잣이 유명합니다.
- **초당두부**: 강릉의 대표적인 음식 특산물입니다.

강원도의 대표적인 특산물은 다음과 같습니다.

- **감자**: 평창·강릉·춘천 등지의 감자
- **옥수수**: 홍천·정선·강릉의 찰옥수수
- **메밀**: 평창 봉평 메밀, 메밀국수·메밀전병
- **황태**: 인제 용대리 황태
- **오징어**: 강릉·속초·울릉 인근 동해안의 오징어
- **대게·붉은대게**: 속초·삼척·고성 등 동해안 지역
- **한우**: 횡성한우
- **더덕·곤드레·산나물**: 정선·평창·홍천 등 산간 지역
- **잣**: 홍천 잣
- **닭갈비·막국수**: 춘천의 대표 향토 음식

특히 **감자, 옥수수, 메밀, 황태, 횡성한우**가 강원도를 대표하는 특산물로 많이 알려져 있습니다.


### Sequential Chain

In [4]:
prompt1 = PromptTemplate.from_template('다음 내용을 한글로 번역하세요. {eng_text}')
prompt2 = PromptTemplate.from_template('다음 내용을 요약하세요. {kor_text}')

llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

# 번역 체인
chain1 = prompt1 | llm

eng_text = """
One limitation of LLMs is their lack of contextual information (e.g., access to some specific documents or emails). You can combat this by giving LLMs access to the specific external data.
For this, you first need to load the external data with a document loader. LangChain provides a variety of loaders for different types of documents ranging from PDFs and emails to websites and YouTube videos.
"""
print(chain1.invoke(eng_text))

chain2 = prompt2 | llm | output_parser
kor_text = """
LLM의 한 가지 한계는 특정 문서나 이메일과 같은 맥락 정보를 갖고 있지 않다는 점입니다. 이를 해결하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.
이를 위해서는 먼저 문서 로더를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF, 이메일부터 웹사이트, 유튜브 영상에 이르기까지 다양한 유형의 문서를 위한 여러 종류의 로더를 제공합니다.
"""
print(chain2.invoke(kor_text))

content='LLM의 한 가지 한계는 맥락 정보가 부족하다는 점입니다. 예를 들어 특정 문서나 이메일에 접근할 수 없습니다. 이러한 한계는 LLM이 특정 외부 데이터에 접근할 수 있도록 함으로써 극복할 수 있습니다.\n\n이를 위해 먼저 문서 로더를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF와 이메일부터 웹사이트 및 YouTube 동영상에 이르기까지 다양한 유형의 문서를 지원하는 여러 로더를 제공합니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 97, 'total_tokens': 209, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHHwEU84G8pmjtWnrkmRBFD5C4sxh', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0408a-dc21-7543-9701-0b9ee8ce9a0c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens'

In [5]:
# Sequential Chain
chain = chain1 | chain2
print(chain.invoke({'eng_text': eng_text}))

LLM은 특정 문서나 이메일 등 외부 맥락에 접근하지 못하는 한계가 있습니다. 이를 해결하려면 외부 데이터를 LLM에 연결해야 하며, LangChain에서는 문서 로더를 통해 PDF, 이메일, 웹사이트, YouTube 등 다양한 형식의 데이터를 불러올 수 있습니다.


### Conditional Chain

In [6]:
from langchain_core.runnables import RunnableBranch  # 조건에 따라 체인을 분기 실행해주는 Runnable

llm = init_chat_model('gpt-5.6-luna')

math_prompt = PromptTemplate.from_template('다음 문제를 풀어주세요. 단계적인 풀이를 중간 풀이과정과 함께 작성해주세요. {question}')
math_chain = math_prompt | llm | output_parser

default_prompt = PromptTemplate.from_template('당신은 친절하고, 감성적이며 공감능력이 좋은 챗봇입니다. 다음 질문에 답변해주세요. {question}')
default_chain = default_prompt | llm | output_parser

# math_chain 선택 함수 (질문에 계산 또는 calc가 포함되면 수학 체인 선택)
def is_math_question(input_dict: dict) -> bool:
    question: str = input_dict.get('question', '')  # 입력받은 dict에서 question 키의 값을 추출(없으면 빈 문자열)
    return '계산' in question or 'calc' in question

# 분기 체인
branch_chain = RunnableBranch(
    (is_math_question, math_chain),  # True/False 결과 조건이 True면 math_chain
    default_chain                    # False면 default_chain
)

print(branch_chain.invoke({'question': '125 * 3 + 50 계산해줘.'}))

단계별로 계산하면:

1. \(125 \times 3 = 375\)
2. \(375 + 50 = 425\)

따라서 정답은 **425**입니다.


In [7]:
print(branch_chain.invoke({'question': '나 오늘 우울해. 빵? 밥?'}))

오늘은 **따뜻한 밥** 어때? 🍚  
우울한 날엔 속이 편안해지는 국이나 계란, 김치처럼 익숙한 걸 먹으면 조금 나아질 수 있어. 그래도 빵이 더 당긴다면 빵 먹어도 괜찮아—오늘은 위로가 되는 쪽으로 가자.  

**밥 먹고 달달한 빵 하나**도 좋은 타협이야. 오늘 많이 힘들었지.


### Memory Chain

`RunnableWithMessageHistory`를 사용하여 대화내역을 기억하는 chain을 생성한다.

In [8]:
from langchain_core.chat_history import BaseChatMessageHistory  # LANGCHAIN 대화기록 메모리 저장용 클래스
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage  # 메시지 타입들
from pydantic import BaseModel, Field  # Pydantic 모델(검증/기본값 생성) 도구
from typing import List  # 타입 힌트(List)

# 사용자별 세션 대화내역을 기록하는 클래스
class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    # Field(default_factory=list) : 인스턴스마다 독립적인 messages list를 구성
    messages: List[BaseMessage] = Field(default_factory=list)

    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages)  # 전달받은 메시지들을 기존 리스트 뒤에 추가
    
    def clear(self) -> None:
        self.messages = []  # 저장된 메시지들을 초기화

store = {}  # {session_id: 히스토리 객체(InMemoryHistory)} 저장소

# 세션 ID로 히스토리 객체를 반환
def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()  # 기존 대화내역이 없으면 히스토리 객체 생성해서 store에 추가
    return store[session_id]  # 해당 세션의 히스토리 객체 반환

history1 = get_by_session_id('1')  # 세션 ID '1'의 히스토리 가져오기 (없으면 메모리 공간 생성)
history1.add_messages([AIMessage(content='반갑습니다. Capybara님!')])  # AI 메시지 추가
history1.add_messages([HumanMessage(content='그래~ 나 Cap이야~ 만나서 반갑다!!')])  # 유저메시지 추가
print(f"{history1 = }")  # f"history1 = {history1}"

history2 = get_by_session_id('2')  # 세션 ID '2'의 히스토리 가져오기
print(f"{history2 = }")

history1 = InMemoryHistory(messages=[AIMessage(content='반갑습니다. Capybara님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래~ 나 Cap이야~ 만나서 반갑다!!', additional_kwargs={}, response_metadata={})])
history2 = InMemoryHistory(messages=[])


### 대화 히스토리를 자동으로 누적하는 Memory Chain (RunnableWithMessageHistory)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # 채팅 프롬프트 템플릿 / 히스토리 자리표시자
from langchain_core.runnables import RunnableWithMessageHistory  # 실행시 히스토리를 붙여주는 Runnable

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name='history'),  # 세션별 이전 대화 메시지들이 들어갈 자리
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm

# 히스토리 기능을 chain에 매핑
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,  # sessionID로 히스토리 객체 가져오는 함수 참조
    input_messages_key = 'question',  # 입력 dict에서 question키의 값은 사용자 메시지
    history_messages_key = 'history'  # 프롬프트에서 history 받을 변수명
)

chain_with_history.invoke({
    'domain': 'math',
    'question': '민수는 강아지를 3마리 키우고 있습니다.'
}, config = {  # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '100'  # 어떤 세션 히스토리 사용할지
    }
})

c:\Users\playdata2\LLM\llm_venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AIMessage(content='민수는 강아지 3마리를 키우고 있군요. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 38, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 21, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHHwTZgrQlQNcaQ0kYiDFfQOMo44g', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0408b-177e-7723-b8ce-27c6ca4126fd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 55, 'total_tokens': 93, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 21}})

In [10]:
chain_with_history.invoke({
    'domain': 'math',
    'question': '소라는 고양이를 4마리 키우고 있습니다.'
}, config = {  # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '100'  # 어떤 세션 히스토리 사용할지
    }
})

AIMessage(content='민수와 소라가 키우는 동물은 모두 **7마리**입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 87, 'total_tokens': 161, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 45, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHI5Loaxnp0X7UWH4kITn1Ktbe6We', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04093-7b31-7a72-a014-9591cf6d39a4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 74, 'total_tokens': 161, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 45}})

In [ ]:
store  # 현재 메모리에 저장된 세션별 대화 히스토리

{'1': InMemoryHistory(messages=[AIMessage(content='반갑습니다. Capybara님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래~ 나 Cap이야~ 만나서 반갑다!!', additional_kwargs={}, response_metadata={})]),
 '2': InMemoryHistory(messages=[]),
 '100': InMemoryHistory(messages=[HumanMessage(content='민수는 강아지를 3마리 키우고 있습니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='민수는 강아지 3마리를 키우고 있군요. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 38, 'total_tokens': 93, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 21, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl

### ChatMessageHistory

In [12]:
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}

# 세션 ID로 히스토리 객체를 반환
def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()  # 기존 대화내역이 없으면 히스토리 객체 생성해서 store에 추가
    return store[session_id]  # 해당 세션의 히스토리 객체 반환

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name='history'),  # 세션별 이전 대화 메시지들이 들어갈 자리
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm | output_parser

# 히스토리 기능을 chain에 매핑
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,  # sessionID로 히스토리 객체 가져오는 함수 참조
    input_messages_key = 'question',  # 입력 dict에서 question키의 값은 사용자 메시지
    history_messages_key = 'history'  # 프롬프트에서 history 받을 변수명
)

chain_with_history.invoke({
    'domain': '심리상담',
    'question': '요즘 갑자기 더워져서 짜증나는데? 나 성격좋은데? 왜이러지?'
}, config = {  # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '200'  # 어떤 세션 히스토리 사용할지
    }
})

c:\Users\playdata2\LLM\llm_venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


'그럴 수 있어요. **성격이 나빠진 게 아니라, 더위가 몸과 뇌에 부담을 줘서 짜증이 쉽게 올라오는 것**일 수 있습니다.\n\n더우면 체온 조절에 에너지를 쓰고, 잠의 질이 떨어지거나 탈수·피로가 생기면서 감정 조절 여유가 줄어들어요. 땀, 끈적임, 소음 같은 감각 자극도 스트레스를 키울 수 있고요.\n\n도움 되는 방법은:\n\n- 물을 조금씩 자주 마시기  \n- 시원한 곳에서 잠깐 쉬며 체온 낮추기  \n- 수면 환경을 시원하고 어둡게 유지하기  \n- 짜증이 날 때 바로 반응하기보다 “지금 더워서 예민해졌을 수 있어”라고 한 박자 두기  \n- 카페인·술을 줄이고 가벼운 식사하기  \n\n다만 더위와 무관하게 **기분 변화가 몇 주 이상 지속되거나, 잠을 거의 못 자고 지나치게 들뜨거나, 우울·불안이 심해지는 경우**에는 상담이나 진료를 받아보는 게 좋아요. 지금 반응만으로 성격 문제라고 볼 필요는 없습니다.'

In [13]:
chain_with_history.invoke({
    'domain': '심리상담',
    'question': '그럼 니가 날씨를 좋게 만들어주면 되잖아'
}, config = {  # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '200'  # 어떤 세션 히스토리 사용할지
    }
})

'그러게요, 제가 날씨까지 조절할 수 있으면 바로 선선하게 바꿔드릴 텐데요 😅  \n대신 **체감온도라도 낮추는 방법**은 같이 찾아드릴 수 있어요.\n\n지금 바로는 시원한 물 마시고, 목·손목·얼굴에 찬물 대고, 선풍기나 에어컨을 직접 몸 전체보다 **목·겨드랑이 주변**으로 약하게 틀어보세요. 그리고 오늘은 “내가 왜 이렇게 예민하지?”보다 “날씨가 나를 좀 괴롭히는구나” 하고 짜증을 잠깐 허용해도 괜찮아요.'

In [ ]:
store

{'200': InMemoryChatMessageHistory(messages=[HumanMessage(content='요즘 갑자기 더워져서 짜증나는데? 나 성격좋은데? 왜이러지?', additional_kwargs={}, response_metadata={}), AIMessage(content='그럴 수 있어요. **성격이 나빠진 게 아니라, 더위가 몸과 뇌에 부담을 줘서 짜증이 쉽게 올라오는 것**일 수 있습니다.\n\n더우면 체온 조절에 에너지를 쓰고, 잠의 질이 떨어지거나 탈수·피로가 생기면서 감정 조절 여유가 줄어들어요. 땀, 끈적임, 소음 같은 감각 자극도 스트레스를 키울 수 있고요.\n\n도움 되는 방법은:\n\n- 물을 조금씩 자주 마시기  \n- 시원한 곳에서 잠깐 쉬며 체온 낮추기  \n- 수면 환경을 시원하고 어둡게 유지하기  \n- 짜증이 날 때 바로 반응하기보다 “지금 더워서 예민해졌을 수 있어”라고 한 박자 두기  \n- 카페인·술을 줄이고 가벼운 식사하기  \n\n다만 더위와 무관하게 **기분 변화가 몇 주 이상 지속되거나, 잠을 거의 못 자고 지나치게 들뜨거나, 우울·불안이 심해지는 경우**에는 상담이나 진료를 받아보는 게 좋아요. 지금 반응만으로 성격 문제라고 볼 필요는 없습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그럼 니가 날씨를 좋게 만들어주면 되잖아', additional_kwargs={}, response_metadata={}), AIMessage(content='그러게요, 제가 날씨까지 조절할 수 있으면 바로 선선하게 바꿔드릴 텐데요 😅  \n대신 **체감온도라도 낮추는 방법**은 같이 찾아드릴 수 있어요.\n\n지금 바로는 시원한 물 마시고, 목·손목·얼굴에 찬물 대고, 선풍기나 에어컨을 직접 몸 전체보다 **목·겨드랑이 주변**으로 약하게 틀어보세요. 그리고 오늘

##### 세션(메모리) 방식의 문제점
- 메모리 저장이라 영속성이 없음
	- 서버 재시작/재배포 하면 store가 날아가서 히스토리도 같이 사라짐
- 세션 식별이 끊기기 쉬움
	- 쿠키/세션ID가 유지되지 않으면 같은 사람인지 매칭이 안 됨
- 스케일 아웃(서버 여러 대)에서 깨짐
	- A서버 메모리에 저장된 히스토리를 B서버는 모름 → 대화가 끊김

그래서 보통 이렇게 구성한다.
- SQLite/Redis/RDB 같은 저장소에 대화 내역을 저장해서
	- 사용자가 재접속해도 user_id 또는 thread_id로 복원
- 프롬프트에는 보통
	- 최근 N턴 + 요약 형태로 넣어서 비용/토큰도 관리